# 面试问题：Prompt、RAG、微调和 Tool Calling 应该怎样选择？

**一句话回答**：先判断缺口属于“指令表达、外部知识、稳定行为，还是可验证动作”。Prompt 是所有方案的低成本基线；知识需要更新、权限或引用时用 RAG；输出风格/任务行为稳定且有足够样本时考虑 SFT；实时查询、精确计算或产生副作用必须调用受控工具。它们不是四选一，常见生产形态是 Prompt + RAG + Tool，必要时再用 SFT 固化行为。

本 Notebook 把口头取舍写成可执行决策合同、成本模型、离线实验和发布门禁，避免用“效果不好就微调”回答。

In [ ]:
from dataclasses import dataclass
import hashlib, json, math
import numpy as np

METHODS103=("prompt","rag","sft","tool")
SEED103=10301; rng103=np.random.default_rng(SEED103)
assert len(METHODS103)==4 and len(set(METHODS103))==4
assert SEED103==10301
assert hashlib.sha256(b"rag").hexdigest()!=hashlib.sha256(b"sft").hexdigest()

## 1. 先把任务缺口写成合同

需要描述知识是否外部/私有、更新频率、是否要求引用、行为是否稳定、是否有高质量训练样本、是否需要实时世界状态或写操作，以及延迟/成本/风险上限。没有这些变量，任何“RAG 还是微调”的答案都只是偏好。

In [ ]:
@dataclass(frozen=True)
class Task103:
    name:str; external_knowledge:bool=False; freshness_hours:int=10**9; citations:bool=False; stable_behavior:bool=False; examples:int=0; realtime:bool=False; side_effect:bool=False; exact_compute:bool=False
    def __post_init__(self):
        if not self.name or self.freshness_hours<0 or self.examples<0: raise ValueError("task_contract")
policy103=Task103("回答最新内部制度",True,24,True)
style103=Task103("固定品牌语气",stable_behavior=True,examples=3000)
booking103=Task103("查询余票并下单",realtime=True,side_effect=True)
assert policy103.external_knowledge and policy103.citations
assert style103.examples==3000 and booking103.side_effect
try: Task103("",examples=-1); raise AssertionError("bad task accepted")
except ValueError as e: assert str(e)=="task_contract"

## 2. Prompt 是基线，不是“低级方案”

先用明确角色、输入边界、少量示例、输出 schema 与拒答条件验证基础模型是否已经具备能力。Prompt 改动快、无需训练，但不能可靠注入超出上下文的知识，也不会让参数记住新事实。基线还提供后续 RAG/SFT 的真实增益分母。

In [ ]:
def prompt_contract103(instruction,context,question):
    if not instruction or not question: raise ValueError("prompt_contract")
    return {"system":instruction,"untrusted_context":context,"user":question,"output_schema":{"answer":"string","citations":"list[string]"},"on_missing_evidence":"abstain"}
base_prompt103=prompt_contract103("仅根据证据回答","制度第3条：退款期7天","退款期多久？")
assert set(base_prompt103)=={"system","untrusted_context","user","output_schema","on_missing_evidence"}
assert base_prompt103["on_missing_evidence"]=="abstain"
try: prompt_contract103("","x","q"); raise AssertionError("empty instruction accepted")
except ValueError as e: assert str(e)=="prompt_contract"

## 3. RAG 解决的是可检索知识与证据问题

文档频繁变化、按租户授权、必须给出处或不能进入权重时，RAG 通常优先。它把错误拆成检索覆盖率、重排和生成忠实度，但引入索引版本、ACL、延迟与上下文预算。检索不到时应该拒答，而不是依赖模型“补全”。

In [ ]:
docs103=[{"id":"p1","tenant":"A","version":3,"text":"退款期为7天"},{"id":"p2","tenant":"B","version":4,"text":"退款期为30天"}]
def retrieve103(query,tenant,k=2):
    terms=set(query); scored=[]
    for d in docs103:
        if d["tenant"]!=tenant: continue
        score=len(terms&set(d["text"])); scored.append((score,d))
    return [d for s,d in sorted(scored,key=lambda z:(-z[0],z[1]["id"]))[:k] if s>0]
hit103=retrieve103("退款期", "A")
assert [d["id"] for d in hit103]==["p1"]
assert all(d["tenant"]=="A" for d in hit103)
assert retrieve103("不存在", "A")==[]

## 4. SFT 更适合稳定行为，不适合频繁更新事实

SFT 可固化格式、语气、领域任务策略和工具选择习惯，并降低长 few-shot prompt 成本；代价是数据治理、训练/评估、遗忘与模型版本管理。知识会频繁变化时，把事实烙进权重既难更新也难引用，通常仍需 RAG/Tool。

In [ ]:
def sft_readiness103(task,min_examples=500):
    blockers=[]
    if not task.stable_behavior: blockers.append("no_stable_behavior_gap")
    if task.examples<min_examples: blockers.append("insufficient_examples")
    if task.freshness_hours<24*30: blockers.append("facts_change_too_fast")
    return {"ready":not blockers,"blockers":blockers}
assert sft_readiness103(style103)["ready"]
assert not sft_readiness103(policy103)["ready"] and "facts_change_too_fast" in sft_readiness103(policy103)["blockers"]
assert "insufficient_examples" in sft_readiness103(Task103("分类",stable_behavior=True,examples=30))["blockers"]

## 5. Tool 用于读取实时状态、精确计算和执行动作

库存、账户余额、天气、SQL、计算器或发邮件都不应靠语言模型参数猜测。模型只提出结构化调用意图，宿主程序做 schema、鉴权、幂等、超时和审批；高风险副作用不能让模型自批自执行。Tool 返回值仍是外部不可信数据。

In [ ]:
TOOL_POLICY103={"calculator":{"read":True,"approval":False},"inventory":{"read":True,"approval":False},"place_order":{"read":False,"approval":True}}
def authorize_tool103(name,user_scopes,approved=False):
    if name not in TOOL_POLICY103: return False,"unknown_tool"
    if name not in user_scopes: return False,"missing_scope"
    if TOOL_POLICY103[name]["approval"] and not approved: return False,"approval_required"
    return True,"allowed"
assert authorize_tool103("calculator",{"calculator"})==(True,"allowed")
assert authorize_tool103("place_order",{"place_order"})==(False,"approval_required")
assert authorize_tool103("shell",{"shell"})==(False,"unknown_tool")

## 6. 先用规则产生候选组合，再用数据决策

规则只负责排除明显不合适的方案：实时/副作用加 Tool，外部新知识/引用加 RAG，稳定行为且样本充足加 SFT；Prompt 永远保留。组合并不意味着所有组件都参与每个请求，路由器可按意图跳过无关链路。

In [ ]:
def architecture103(task):
    parts=["prompt"]
    if task.external_knowledge or task.citations: parts.append("rag")
    if task.stable_behavior and task.examples>=500: parts.append("sft")
    if task.realtime or task.side_effect or task.exact_compute: parts.append("tool")
    return tuple(parts)
assert architecture103(policy103)==("prompt","rag")
assert architecture103(style103)==("prompt","sft")
assert architecture103(booking103)==("prompt","tool")

## 7. 用质量—延迟—成本—风险做 Pareto 比较

每个候选在同一黄金集与流量回放上测任务质量、引用正确率、p95、单请求成本和高风险失败。加权分数只用于同一业务权重下排序；若方案违反硬门槛，即使平均分高也不可发布。下面演示先过滤硬约束，再算效用。

In [ ]:
candidates103=[{"name":"prompt","quality":.68,"p95":.3,"cost":.002,"risk":.08},{"name":"prompt+rag","quality":.86,"p95":.7,"cost":.006,"risk":.03},{"name":"prompt+sft","quality":.79,"p95":.35,"cost":.004,"risk":.06}]
def rank103(rows,max_p95,max_risk):
    feasible=[r for r in rows if r["p95"]<=max_p95 and r["risk"]<=max_risk]
    return sorted(feasible,key=lambda r:r["quality"]-5*r["cost"]-.1*r["p95"],reverse=True)
ranked103=rank103(candidates103,1.,.05)
assert [r["name"] for r in ranked103]==["prompt+rag"]
assert rank103(candidates103,.5,.05)==[]
assert all(r["risk"]<=.05 for r in ranked103)

## 8. 决策要能被实验推翻

先锁定数据快照与 baseline，再做单变量 ablation：只加检索、只加训练、只加工具，报告均值和关键 slice。发布门禁同时检查质量、拒答、权限、延迟和成本；上线后保留 shadow/canary 与快速回滚，而不是把架构选择当永久结论。

In [ ]:
baseline103={"quality":.70,"unsafe":.015,"p95":.32}; challenger103={"quality":.82,"unsafe":.008,"p95":.48}
gates103={"min_quality_gain":.05,"max_unsafe":.01,"max_p95":.5}
passed103=challenger103["quality"]-baseline103["quality"]>=gates103["min_quality_gain"] and challenger103["unsafe"]<=gates103["max_unsafe"] and challenger103["p95"]<=gates103["max_p95"]
manifest103={"schema":1,"task":"internal_policy_qa","architecture":architecture103(policy103),"dataset":"gold-v4","baseline":"prompt-v2","gates":gates103}; digest103=hashlib.sha256(json.dumps(manifest103,sort_keys=True).encode()).hexdigest()
assert passed103
assert manifest103["architecture"]==("prompt","rag") and manifest103["dataset"]=="gold-v4"
assert len(digest103)==64

## 面试总结

回答顺序建议是：**诊断缺口 → Prompt baseline → 新鲜/私有/可引用知识用 RAG → 稳定行为用 SFT → 实时/精确/副作用用 Tool → 组合路由 → 同预算 ablation → 安全发布**。最重要的观点是“知识、行为和动作是不同问题”，不能用一种技术包打天下。

延伸阅读：[RAG 原始论文](https://arxiv.org/abs/2005.11401)、[LoRA](https://arxiv.org/abs/2106.09685)、[Toolformer](https://arxiv.org/abs/2302.04761)。